# Topical Lectures, Controls 
## Drone control, part 5: model-based control

Andreas Freise, 27.05.2026

The previous notebooks built a cascade of three PID controllers and raced
the drone through a track. In this notebook we use the drone's equations
of motion directly to compute the tilt and thrust required for any desired
acceleration. The result is a faster, cleaner flight path with very little
gain tuning.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import time
import module
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

%matplotlib inline

# Don't forget to set the name string to your own name
MyName = "Andreas"


## Why model-based control

The cascade in notebook 3 looked like

* outer y PID -> desired tilt
* outer z PID -> thrust correction
* inner phi PD -> torque command

Three things were uncomfortable about it.

1. The y PID output had no units. We clipped it at $\pm 0.2\pi$ to make
   it look like a tilt command, but the connection between gain choice
   and tilt aggressiveness was only correct by accident.
2. When the drone tilted to move sideways, the vertical lift dropped as
   $\cos(\phi)$ and the z controller had to chase the resulting sag.
   We patched this with an explicit $V_{\mathrm{hover}}/\cos(\phi)$
   feedforward.
3. y and z were treated as decoupled, but the drone dynamics couple them
   strongly through $\phi$.

We can do better by inverting the drone's equations of motion analytically.


## Inverting the dynamics

The 2D drone has

$$
\ddot{y} = -\sin(\phi)\,\frac{F}{M}
\qquad
\ddot{z} =  \cos(\phi)\,\frac{F}{M} - g
$$

Given a desired pair $(\ddot{y}_{des}, \ddot{z}_{des})$ we can solve
algebraically for the unique $(\phi, F)$ that produces it. Rearrange

$$
-F\sin(\phi) = M\,\ddot{y}_{des}
\qquad
 F\cos(\phi) = M\,(\ddot{z}_{des} + g)
$$

Square and add to get $F$, divide to get $\phi$:

$$
F   = M\,\sqrt{\ddot{y}_{des}^{\,2} + (\ddot{z}_{des} + g)^{2}}
\qquad
\phi = \arctan2(-\ddot{y}_{des},\;\ddot{z}_{des} + g)
$$

This pair of formulas is `module.accel_to_phi_F(a_y, a_z)`. The control
structure becomes:

1. Two PIDs that emit desired accelerations $\ddot{y}_{des}, \ddot{z}_{des}$.
2. The algebraic conversion to $(\phi_{des}, F_{des})$.
3. A phi PD that tracks $\phi_{des}$.
4. $F_{des}$ as the common-mode thrust command.

The $\pm 0.2\pi$ clip and the $V_{\mathrm{hover}}/\cos(\phi)$ feedforward
both disappear. The $+g$ term inside $F_{des}$ provides the hover thrust
automatically; $\arctan2$ never produces a runaway tilt.

The one subtlety we keep is a safety wedge on the desired acceleration:
if $\ddot{z}_{des}$ approaches $-g$ from above, $\ddot{z}_{des} + g$
goes through zero and $\arctan2$ swings to $\pm\pi/2$, which would tip
the drone onto its side. We clip $\ddot{z}_{des} \ge -0.9g$ and bound
the resulting tilt to $\phi_{max} = \pi/4$.


In [ ]:
# Measured parameters for the Crazyflie-scale drone (SI units throughout).
# These would normally come from the student2 system-identification notebook.
V_left_offset = -0.0027   # throttle units; cancels left/right asymmetry
V_hover = 0.5297          # per-motor throttle at hover (= M*g/2 / amp)
V_max = 1.0               # throttle range upper bound
V_min = 0                 # motors are single-direction

# Common-mode / differential decomposition for motor mixing.
# Total motor command must stay in [V_min, V_max]; the differential is
# bounded by the headroom that the common-mode thrust leaves on each side.

def set_v(drone, _V_left, _V_right):
    V_left = V_hover + _V_left + V_left_offset
    V_right = V_hover + _V_right
    fb_z = (V_left + V_right) / 2
    fb_phi = (V_left - V_right) / 2
    fb_z = max(V_min, min(fb_z, V_max))
    headroom = min(fb_z - V_min, V_max - fb_z)
    fb_phi = max(-headroom, min(fb_phi, headroom))
    drone.set_V(fb_z + fb_phi, fb_z - fb_phi)

def set_v_nohover(drone, _V_left, _V_right):
    V_left = _V_left + V_left_offset
    V_right = _V_right
    fb_z = (V_left + V_right) / 2
    fb_phi = (V_left - V_right) / 2
    fb_z = max(V_min, min(fb_z, V_max))
    headroom = min(fb_z - V_min, V_max - fb_z)
    fb_phi = max(-headroom, min(fb_phi, headroom))
    drone.set_V(fb_z + fb_phi, fb_z - fb_phi)

def zphi2V(fb_z, fb_phi):
    V_left = fb_z + fb_phi
    V_right = fb_z - fb_phi
    return V_left, V_right


## Controllers

We instantiate three control layers, each as a `PIDController`:

* Outer position PIDs (one per axis): produce a desired velocity, capped at `v_max` so the drone never asks for an unreachable speed.
* Inner velocity PIDs: convert the velocity error into a desired acceleration. The cascade matches what the real Crazyflie firmware does, and it caps the drone's speed regardless of how far the position setpoint is.
* Phi PD: tracks the tilt commanded by the inversion.

The PID class is the same one we used for Layer A; only the role of each instance differs.

In [ ]:
# Cascade structure: position -> velocity (capped) -> acceleration ->
# inversion. The outer two layers are the same shape as student4b's
# velocity cascade; the inner conversion to (tilt, thrust) is the new
# model-based step.
y_pos_ctrl = module.PIDController(Kp=5.0, Ki=0,    Kd=0)
z_pos_ctrl = module.PIDController(Kp=5.0, Ki=0,    Kd=0)
y_vel_ctrl = module.PIDController(Kp=5.0, Ki=0.3,  Kd=0)
z_vel_ctrl = module.PIDController(Kp=5.0, Ki=0.3,  Kd=0)
phi_ctrl   = module.PIDController(Kp=0.3, Ki=0,    Kd=0.06)

v_max       = 2.5             # m/s, the velocity cap
phi_max_des = np.pi / 6       # 30 degrees, the safety wedge half-angle
az_min      = -0.7 * 9.8      # m/s^2, max downward command we will ask for

## Race the track

Same five-target track as `student4b_racing_noninteractive.ipynb`, so
you can compare race times and flight paths directly.


In [ ]:
drone = module.Drone(name=MyName, wind=True)

targets = np.array([
    [ 1.0,  1.5, 0.1],
    [ 1.0, -1.5, 0.1],
    [-1.0, -1.5, 0.1],
    [-1.0,  1.5, 0.1],
    [   0,    0, 0.1]
])
drone.set_targets(targets)

N = 200 * 60
Nidx = N - 1
results = np.zeros([N, 14])
V_left = V_right = 0

for ctrl in (y_pos_ctrl, z_pos_ctrl, y_vel_ctrl, z_vel_ctrl, phi_ctrl):
    ctrl.reset()

for i in range(N):
    state = drone.update()
    results[i, 0:12] = module.state_to_array(state)
    results[i, 12:14] = [V_left, V_right]

    idx = drone.target_idx
    if idx == drone.num_targets:
        Nidx = i
        break
    y0 = targets[idx, 0]
    z0 = targets[idx, 1]

    # Outer position loops -> desired velocity, saturated at +-v_max.
    dy_des = float(np.clip(-y_pos_ctrl(state.y, state.dy, y0), -v_max, v_max))
    dz_des = float(np.clip(-z_pos_ctrl(state.z, state.dz, z0), -v_max, v_max))

    # Velocity loops -> desired acceleration along each axis.
    a_y_des = -y_vel_ctrl(state.dy, 0, dy_des)
    a_z_des = -z_vel_ctrl(state.dz, 0, dz_des)

    # Safety wedge so phi_des stays within [-phi_max_des, +phi_max_des].
    a_z_des = max(a_z_des, az_min)
    a_y_bound = (a_z_des + drone.g) * np.tan(phi_max_des)
    a_y_des = float(np.clip(a_y_des, -a_y_bound, a_y_bound))

    # Algebraic inversion of the dynamics: (a_y, a_z) -> (phi, F).
    phi_des, F_des = module.accel_to_phi_F(a_y_des, a_z_des, M=drone.M, g=drone.g)

    # Inner phi PD tracks phi_des.
    phi_fb = phi_ctrl(state.phi, state.dphi, phi_des)

    # Total thrust split across the two motors as a common-mode V signal.
    fb_z = (F_des / 2.0) / drone.V_left_amp
    V_left, V_right = zphi2V(fb_z, phi_fb)
    set_v_nohover(drone, V_left, V_right)

print(f"Total time: {results[Nidx, 0]:.2f} out of {N/60.0:.2f} seconds")

In [ ]:
# Trajectory plot
start = 0
y = results[start:Nidx+1, 1]
z = results[start:Nidx+1, 2]

fig, ax = plt.subplots(1, figsize=(12, 6))
for _t in targets:
    ax.add_patch(plt.Circle((_t[0], _t[1]), _t[2], lw=1, color='red', alpha=0.5, fill=False))
ax.plot(y, z, 'k')
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.grid()
plt.tight_layout()


## Inline replay of the race

The HTML5 video plays inline in any notebook, including Google Colab.
Run student4b first and compare the two videos side by side.


In [ ]:
module.render_replay(drone, results[:Nidx + 1])


## Compared to the cascade

Running the same track in `student4b_racing_noninteractive.ipynb` with the velocity-loop cascade gives about 17 seconds. The model-based controller in this notebook completes the same track in about 10 seconds. The improvement is most visible on the diagonal moves: the inversion picks the correct tilt for the desired acceleration directly, so the drone does not sag in z when it tilts to move in y.

The outer cascade structure (position $\rightarrow$ velocity $\rightarrow$ acceleration) is the same as in student4b. The novelty here is the bottom layer: the dimensionless tilt clip is replaced by an algebraic conversion from desired acceleration to (tilt, thrust). The `+g` term inside `F_des` provides the hover thrust automatically; `arctan2` never produces a runaway tilt, so the $\pm 0.2\pi$ clip disappears.

What this notebook does *not* do is plan a trajectory between waypoints. The position setpoint still jumps from one corner to the next, which means the drone overshoots at every corner. The next notebook smooths that out with a quintic trajectory that the drone flies *through* every intermediate waypoint at non-zero speed.